# Fig. 7 and Table 4: physical consequences of Forms 1-3

Density, radial dispersion, log slope and anisotropy of the bound particles of four halos, against
the self-consistent isotropic models of Forms 1-3 (Table 3), out to $20\,r_s$. Everything is in each halo's structural units: $r/r_s$, $\rho/\rho_s$,
$\sigma/\sqrt{\Psi(0)}$. Also produces Table 4: $\gamma_{in}$, $\gamma(R_{vir})$ and the
formal $\gamma_{out} = \alpha + 3/2$.

Model, with $q = \Psi(0)/(G r_s^2 \rho_s)$ and $\hat\Psi = \Psi/\Psi(0)$:
$\hat\rho = 4\sqrt2\pi C q^{3/2} I_{1/2}(\hat\Psi)$, $\hat\sigma_r^2 = \tfrac23 I_{3/2}/I_{1/2}$,
$\hat\nabla^2\hat\Psi = -(4\pi/q)\hat\rho$, with $I_k = \int_0^{\hat\Psi}\phi(\varepsilon)(\hat\Psi-\varepsilon)^k d\varepsilon$,
integrated outwards from $\hat\Psi(0) = 1$ on the singular branch. $C$ and $q$ enter only as $C\sqrt q$,
which rescales $r$, so Halo 2's $q$ is used throughout.

Run from `paper_figures/`. The figure is drawn here with matplotlib; for the paper version copy
`fig7/` to the Mac and run `gnuplot fig7.gnu` there.

In [ ]:
import json
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import mpmath as mp
import numpy as np
import pandas as pd
from scipy.integrate import cumulative_trapezoid, quad, solve_ivp
from scipy.interpolate import PchipInterpolator
from scipy.special import beta as Bfn, gamma as Gfn, hyp2f1

warnings.filterwarnings("ignore", module="scipy.integrate")

sys.path.insert(0, str(Path.cwd().resolve().parents[1]))  # AurigaDFs/, so the package imports
from Empirical_DF_Construction.io.results_paths import halo_results_root
from Empirical_DF_Construction.paper_figures.common import G, bound_particles, density_fit, out_dir

HALO_RESULTS = halo_results_root()   # snapshots ($AURIGA_DATA) are only read if a shell cache is missing
OUT = Path.cwd() / "fig7"
OUT.mkdir(exist_ok=True)

HALOS = [1, 2, 6, 20]          # legend order
REF = 2                        # the models use its q, and R_vir is its R_200m
SOFTENING_KPC = 0.369
MIN_COUNTS = 1000              # particles per shell for sigma and beta
R_MIN_KPC = 3 * SOFTENING_KPC  # innermost converged radius
X_MAX = 20.0                   # outer edge of panels (a)-(d), in r_s
X_OUT = 1e3                    # outer edge of the integration, for the tail check
EDGES = np.logspace(np.log10(SOFTENING_KPC), np.log10(2000), 81)

FCOL = {"form1": "#ff7f0e", "form2": "#d62728", "form3": "#1f77b4"}
HCOL = {2: "#000000", 6: "#2ca02c", 20: "#9467bd", 1: "#8c564b"}   # Halo 2 black and thick

## Forms (Table 3)

Central values and halo-bootstrap percentiles, from `output/forms_params.csv` (written by `table3_fig6_generalised_forms`).
Form 1 and 3 have $\beta = \alpha$; Form 3 has no outer power law.

In [ ]:
P = pd.read_csv(out_dir() / "forms_params.csv", index_col="form")
FORMS = {f"form{k}": dict(C=P.C[k], a=P.alpha[k], b=P.beta[k] if k == 2 else P.alpha[k],
                          e0=P.eps0[k] if k == 3 else 0.0) for k in (1, 2, 3)}
for t, p in FORMS.items():
    print(f"{t}: C = {p['C']:.4e}, alpha = {p['a']:.4f}, beta = {p['b']:.4f}, eps0 = {p['e0']:.5f}")

## Scales

$R_{vir}$ is Halo 2's $R_{200m}$, measured from the particles, so $c = R_{200m}/r_s$.

In [ ]:
def scales(h):
    d = HALO_RESULTS / f"Halo_{h:02d}"
    nfw = density_fit(h)["nfw"]
    psi0 = json.loads((d / "df" / "raw" / "df_raw_with_hubble_flow_meta.json").read_text())["Psi0"]
    return dict(rs=nfw["rs"], rhos=nfw["rhos"], psi0=psi0, q=psi0 / (G * nfw["rhos"] * nfw["rs"] ** 2))


S = {h: scales(h) for h in HALOS}
for h, s in S.items():
    print(f"Halo {h:02d}: r_s = {s['rs']:.2f} kpc, rho_s = {s['rhos']:.3e}, Psi0 = {s['psi0']:.0f}, q = {s['q']:.3f}")

R_VIR = json.loads((HALO_RESULTS / f"Halo_{REF:02d}" / "df" / "raw" / "df_naive_radial_R200m_meta.json")
                   .read_text())["R200m_kpc"]
C_VIR = R_VIR / S[REF]["rs"]
print(f"Halo {REF:02d}: R_vir = R_200m = {R_VIR:.1f} kpc, c = {C_VIR:.2f}")

### $q$ across the suite

If $q_i$ is the same to a few per cent across halos, the $\hat\varepsilon = \varepsilon/\Psi(0)$ and NFW
scalings agree up to a constant, which is what the similarity argument in §3.3 needs.

In [ ]:
Q = {}
for d in sorted(HALO_RESULTS.glob("Halo_[0-9][0-9]")):
    h = int(d.name[-2:])
    try:
        Q[h] = scales(h)["q"]
    except FileNotFoundError as e:
        print(f"Halo {h:02d}: {e.filename} missing, left out")
qs = np.array(list(Q.values()))
np.savetxt(OUT / "q_all.dat", np.column_stack([list(Q), qs]), fmt=["%d", "%.6f"], header="halo q")
print("  ".join(f"{h:02d}:{v:.2f}" for h, v in Q.items()))
print(f"{qs.size} halos: q = {qs.min():.2f}-{qs.max():.2f}, median {np.median(qs):.2f}, "
      f"(max - min)/median = {np.ptp(qs) / np.median(qs):.1%}, std/mean = {qs.std() / qs.mean():.1%}")

## Particles: density, $\sigma_r$ and $\beta$ in radial shells

Shells are cached in `fig7/haloXX_shells.npz` (all radii), so changing `X_MAX` needs no re-read.

In [ ]:
def shells(h):
    cache = OUT / f"halo{h:02d}_shells.npz"
    if not cache.exists():
        p = bound_particles(h)
        x, v, mpart = p["x_kpc"], p["v_kms"], p["m_msun"][0]
        r = np.linalg.norm(x, axis=1)
        vr = np.einsum("ij,ij->i", v, x) / r
        v2 = np.einsum("ij,ij->i", v, v)
        j = np.digitize(r, EDGES) - 1
        n = np.bincount(j[(j >= 0) & (j < len(EDGES) - 1)], minlength=len(EDGES) - 1)
        rho = n * mpart / (4 / 3 * np.pi * (EDGES[1:] ** 3 - EDGES[:-1] ** 3))
        sig, bet = np.full(n.size, np.nan), np.full(n.size, np.nan)
        for i in np.where(n >= MIN_COUNTS)[0]:
            k = j == i
            sr2 = np.var(vr[k])
            sig[i], bet[i] = np.sqrt(sr2), 1 - (np.mean(v2[k]) - np.mean(vr[k] ** 2)) / (2 * sr2)
        np.savez(cache, r=np.sqrt(EDGES[1:] * EDGES[:-1]), n=n, rho=rho, sigma=sig, beta=bet)
    d = dict(np.load(cache))
    s = S[h]
    k = (d["n"] >= MIN_COUNTS) & (d["r"] >= R_MIN_KPC) & (d["r"] / s["rs"] <= X_MAX)
    x, y = d["r"][k] / s["rs"], d["rho"][k] / s["rhos"]
    return dict(x=x, rho=y, slope=-np.gradient(np.log(y), np.log(x)),   # plain finite differences
                sigma=d["sigma"][k] / np.sqrt(s["psi0"]), beta=d["beta"][k])


DATA = {h: shells(h) for h in HALOS}
lo = min(d["x"].min() for d in DATA.values())
for h, d in DATA.items():
    print(f"Halo {h:02d}: r/r_s in [{d['x'].min():.3f}, {d['x'].max():.1f}]")

## Models

$I_k$ in closed form, $\psi^{\alpha+k+1}B(\alpha+1,k+1)\,{}_2F_1(\alpha+1,\beta;\alpha+k+2;\psi)$; Form 3 adds
a bounded correction by quadrature. Near the centre $\hat\rho \simeq K(1-\hat\Psi)^{3/2-\beta}$, which
fixes the singular starting values.

In [ ]:
def I_ratio(psi, a, b, k):
    """int_0^psi e^a (1-e)^-b (psi-e)^k de."""
    return psi ** (a + k + 1) * Bfn(a + 1, k + 1) * hyp2f1(a + 1, b, a + k + 2, psi)


def make_I(p):
    a, b, e0 = p["a"], p["b"], p["e0"]
    if e0 == 0:
        return lambda psi, k: I_ratio(psi, a, b, k)
    g1 = np.exp(-e0)
    fn = lambda e: e ** a * (1 - e) ** (-a) * (np.exp(-e0 / e) - g1) if e > 0 else 0.0
    grid = np.unique(np.concatenate([np.linspace(0, 0.9, 600), 1 - np.logspace(-1, -13, 800)]))
    corr = {k: PchipInterpolator(grid, [quad(fn, 0, g, weight="alg", wvar=(0, k), limit=500)[0]
                                        if g > 0 else 0.0 for g in grid]) for k in (0.5, 1.5)}
    return lambda psi, k: g1 * I_ratio(psi, a, a, k) + corr[k](psi)


def setup(p, q, x0=1e-7):
    """rho(Psi), I_k, the ODE in s = ln r and its singular starting values at r = x0."""
    I = make_I(p)
    a, b, C = p["a"], p["b"], p["C"]
    clip = lambda psi: np.clip(psi, 0, 1 - 1e-13)
    rho = lambda psi: 4 * np.sqrt(2) * np.pi * C * q ** 1.5 * I(clip(psi), 0.5)
    K = (4 * np.sqrt(2) * np.pi * C * q ** 1.5 * np.exp(-p["e0"]) * Bfn(a + 1, 1.5)
         * Gfn(a + 2.5) * Gfn(b - 1.5) / (Gfn(a + 1) * Gfn(b)))
    m = 2 / (b - 0.5)
    c = ((4 * np.pi / q) * K / (m * (m + 1))) ** (1 / (b - 0.5))
    rhs = lambda s, y: [y[1], -y[1] - (4 * np.pi / q) * float(rho(y[0])) * np.exp(2 * s)]
    return rho, I, clip, rhs, [1 - c * x0 ** m, -m * c * x0 ** m]


def solve(p, q, x0=1e-7, x1=X_OUT):
    rho, I, clip, rhs, y0 = setup(p, q, x0)
    sol = solve_ivp(rhs, (np.log(x0), np.log(x1)), y0, method="Radau", rtol=1e-10, atol=1e-13, dense_output=True)
    s = np.linspace(np.log(x0), np.log(x1), 8000)
    psi, dpsi = sol.sol(s)
    r = rho(psi)
    return dict(x=np.exp(s), s=s, psi=psi, dpsi=dpsi, rho=r, q=q,
                slope=-np.gradient(np.log(r), s),
                sigma=np.sqrt(2 / 3 * I(clip(psi), 1.5) / I(clip(psi), 0.5)))


MOD = {t: solve(p, S[REF]["q"]) for t, p in FORMS.items()}
inner = lambda m: (m["x"] >= lo) & (m["x"] <= X_MAX)
for t, m in MOD.items():
    w = inner(m)
    print(f"{t}: r_-2/r_s = {np.interp(2.0, m['slope'][w], m['x'][w]):.3f}, "
          f"rho/rho_s at r_s = {np.interp(1.0, m['x'], m['rho']):.3f}   (NFW: 1, 0.25)")

## Reference profiles: NFW and Halo 2's Einasto fit

In [ ]:
ein = json.loads((HALO_RESULTS / f"Halo_{REF:02d}" / "density" / "density_fit_params_with_hubble_flow.json")
                 .read_text())["einasto"]
einasto = lambda x: ein["rho2"] * np.exp(-2 / ein["alpha"] * ((x * S[REF]["rs"] / ein["r2"]) ** ein["alpha"] - 1)) / S[REF]["rhos"]
nfw = lambda x: 1 / (x * (1 + x) ** 2)
logslope = lambda f, x: -np.gradient(np.log(f(x)), np.log(x))
xg = np.logspace(np.log10(lo), np.log10(X_MAX), 600)

## Numerical checks

1. Poisson in integral form, $\hat\Psi = 1 - q^{-1}\int \hat M/\hat r^2\,d\hat r$, and the isotropic Jeans
   equation $d(\hat\rho\hat\sigma^2)/d\hat r = \hat\rho\,d\hat\Psi/d\hat r$, over the plotted range.
2. Both reuse $I_k$, so they would not catch an error in `hyp2f1`. For $k = 1/2$, $c - a - b = 3/2 - \beta < 0$
   and the start has $1 - \hat\Psi \sim 10^{-8}$: compare with mpmath at 50 digits, at exactly the double scipy sees.
3. Outer behaviour. A Keplerian tail ($\hat\Psi \to \hat M/q\hat r$, slope $\to \alpha + 3/2$) needs
   $\hat\Psi \propto 1/\hat r$ exactly once the mass has converged; with nothing left to tune that would be a
   coincidence, and for $n = \alpha + 3/2 < 5$ the models instead truncate far out (Lane-Emden). The
   $\alpha + 3/2$ line in (c) is drawn only for a form that passes (potential Keplerian to 2%, slope within 0.05).

In [ ]:
KEP = {}
for t, m in MOD.items():
    p = FORMS[t]
    x, s, r = m["x"], m["s"], m["rho"]
    g_in = 2 * (p["b"] - 1.5) / (p["b"] - 0.5)
    M = 4 * np.pi * x[0] ** 3 * r[0] / (3 - g_in) + cumulative_trapezoid(4 * np.pi * x ** 3 * r, s, initial=0)
    w = inner(m)
    dpsi = np.abs(m["psi"][0] - cumulative_trapezoid(M / x / m["q"], s, initial=0) - m["psi"])[w].max()
    jl, jr = np.gradient(r * m["sigma"] ** 2, s), r * m["dpsi"]
    line = f"{t}: Poisson |dPsi| <= {dpsi:.1e}, Jeans rel. residual <= {np.max(np.abs(jl - jr)[w] / np.abs(jr[w])):.1e}"
    if not p["e0"]:   # Form 3 has an exponential cut-off at small eps, no power-law tail
        i = -50       # clear of the one-sided gradient at the edge
        kep, target = M[i] / (m["q"] * x[i]) / m["psi"][i], p["a"] + 1.5
        ok = abs(kep - 1) < 0.02 and abs(m["slope"][i] - target) < 0.05
        line += (f"; at r/r_s = {x[i]:.0f}: Psi_kep/Psi = {kep:.3f}, slope = {m['slope'][i]:.3f} "
                 f"vs alpha + 3/2 = {target:.3f} -> {'Keplerian' if ok else 'not Keplerian'}")
        if ok:
            KEP[t] = target
    print(line)

mp.mp.dps = 50
for t, p in FORMS.items():
    a, b = p["a"], p["b"]
    for k in (0.5, 1.5):
        err = [float(abs(hyp2f1(a + 1, b, a + k + 2, 1 - d) / mp.hyp2f1(a + 1, b, a + k + 2, mp.mpf(1 - d)) - 1))
               for d in (1e-4, 1e-8, 1e-12)]
        print(f"{t}, k = {k}: c - a - b = {k + 1 - b:+.3f}, |scipy/mpmath - 1| at 1 - psi = 1e-4, 1e-8, 1e-12: "
              + ", ".join(f"{e:.1e}" for e in err))

### Truncation radius

Integrate past the plotted range until $\hat\Psi = 0$, where $\rho$ vanishes: the edge of the model. The
models are finite (check 3); this says where. $r_t \propto (C\sqrt q)^{-1/2}$, so it uses Halo 2's $q$ like
everything else. Repeated at a tighter tolerance to show the value is converged.

Forms 1-2 only. Form 3's $\exp(-\varepsilon_0/\varepsilon)$ cut-off makes $\hat\rho$ fall faster than any power as
$\hat\Psi \to 0$, so $\hat\Psi$ levels off near $6\times10^{-3}$ and has no finite edge; beyond $\sim10^5 r_s$ its
$\hat\rho \lesssim 10^{-11}$ is also below the accuracy of the quadrature correction in `make_I`
(irrelevant inside $20\,r_s$, where $\hat\rho > 10^{-4}$).

In [ ]:
def truncation_radius(p, q, rtol, atol, x0=1e-7, x_stop=1e12):
    rho, I, clip, rhs, y0 = setup(p, q, x0)
    edge = lambda s, y: y[0]
    edge.terminal, edge.direction = True, -1
    sol = solve_ivp(rhs, (np.log(x0), np.log(x_stop)), y0, method="Radau", rtol=rtol, atol=atol, events=edge)
    return np.exp(sol.t_events[0][0]) if sol.t_events[0].size else np.inf


R_T = {}
for t, p in FORMS.items():
    if p["e0"]:
        continue
    xt = [truncation_radius(p, S[REF]["q"], *tol) for tol in ((1e-10, 1e-16), (1e-12, 1e-18))]
    R_T[t] = xt[1]
    print(f"{t}: Psi = 0 at r/r_s = {xt[1]:.3e} ({xt[1] * S[REF]['rs'] / 1e3:.3g} Mpc for Halo 2); "
          f"looser tolerance {xt[0]:.3e}, change {abs(xt[0] / xt[1] - 1):.1e}")

## Why the models' $\hat\sigma_r$ is low: the anisotropy check

For each halo, against its measured $\sigma_r$ (all as ratios):

- **Jeans, measured $\beta$**: $\sigma_r$ from the anisotropic Jeans equation with the measured $\rho$, $\Psi$ and $\beta(r)$.
  Close to 1 means the particles, their potential and $\beta$ are consistent.
- **Jeans, $\beta = 0$**: the same, for an isotropic halo with identical $\rho$ and $\Psi$. The baseline for any $f(E)$.
- **own $f(E)$**: $\sigma_r$ from integrating the halo's own measured DF in its potential.
- **model**: the self-consistent Forms 1-3 (their mean; they agree to 1%).
- **local**: $\sqrt{1 - 2\beta/3}$, the isotropic $\sigma = \sqrt{\langle v^2\rangle/3}$ with $\langle v^2\rangle = (3-2\beta)\sigma_r^2$
  at the same radius. It ignores that the $2\beta/r$ term of the Jeans equation accumulates over all radii outside $r$.

$\Psi(r)$ is rebuilt from the measured density exactly as the DF pipeline builds it ($\Phi(\infty) = 0$).
Medians over $0.05$-$2\,r_s$ and $0.05$-$20\,r_s$, and the value at $r_s$; $\beta$ is also given within each halo's $R_{200m}$.

In [ ]:
from Empirical_DF_Construction.core.spherical import potential_from_rho
from Empirical_DF_Construction.io.results_paths import density_dir, df_raw_dir


def sigma_ratios(h):
    s = S[h]
    sh = dict(np.load(OUT / f"halo{h:02d}_shells.npz"))
    k = (sh["n"] >= MIN_COUNTS) & (sh["r"] >= R_MIN_KPC)
    r, sig, beta = sh["r"][k], sh["sigma"][k], sh["beta"][k]

    dens = pd.read_csv(density_dir(h) / "density_with_hubble_flow.csv")
    meta = json.loads((df_raw_dir(h) / "df_raw_with_hubble_flow_meta.json").read_text())
    rg = np.logspace(np.log10(dens.r_kpc.iloc[0]), np.log10(meta["rmax_kpc"]), 8000)
    rho = np.exp(np.interp(np.log(rg), np.log(dens.r_kpc), np.log(dens.rho_msun_kpc3)))
    psi, M = potential_from_rho(rg, rho, Gval=G)

    def jeans(b):   # rho sigma_r^2 w(r) = int_r^oo rho G M / r'^2 w dr',  w = exp(int 2 beta / r dr)
        w = np.exp(cumulative_trapezoid(2 * np.interp(rg, r, b) / rg, rg, initial=0))
        I = cumulative_trapezoid(rho * G * M / rg ** 2 * w, rg, initial=0)
        return np.sqrt(np.interp(r, rg, (I[-1] - I) / (rho * w)))

    df = pd.read_csv(df_raw_dir(h) / "df_raw_with_hubble_flow.csv").dropna()
    E, f = df.eps_hat.to_numpy() * s["psi0"], 10 ** df.log10f.to_numpy()

    def Ik(p, k):   # int_0^p f(E) (p - E)^k dE
        e = np.concatenate([[0], E[E < p], [p]])
        fe = np.concatenate([[0], f[E < p], [np.interp(p, E, f)]])
        return np.trapezoid(fe * (p - e) ** k, e)

    own_f = np.array([np.sqrt(2 / 3 * Ik(p, 1.5) / Ik(p, 0.5)) for p in np.interp(r, rg, psi)])
    model = np.mean([np.interp(r / s["rs"], m["x"], m["sigma"]) for m in MOD.values()], axis=0) * np.sqrt(s["psi0"])
    return pd.DataFrame({"x": r / s["rs"], "beta": beta,
                         "Jeans, measured beta": jeans(beta) / sig, "Jeans, beta = 0": jeans(0 * beta) / sig,
                         "own f(E)": own_f / sig, "model": model / sig, "local": np.sqrt(1 - 2 * beta / 3)})


rows = []
for h in HALOS:
    t = sigma_ratios(h)
    r200 = json.loads((df_raw_dir(h) / "df_naive_radial_R200m_meta.json").read_text())["R200m_kpc"] / S[h]["rs"]
    for lo_, hi_ in ((0.05, 2), (0.05, 20)):
        w = t.x.between(lo_, hi_)
        rows.append({"halo": h, "range": f"{lo_}-{hi_} r_s", **t[w].drop(columns="x").median().to_dict()})
    at = {c: np.interp(0.0, np.log(t.x), t[c]) for c in t.columns.drop("x")}
    rows.append({"halo": h, "range": "at r_s", **at})
    rows.append({"halo": h, "range": "within R_200m", "beta": t.beta[t.x <= r200].median()})
SIGMA = pd.DataFrame(rows).set_index(["range", "halo"]).sort_index()
SIGMA.to_csv(OUT / "sigma_offset.csv")
SIGMA.round(3)

## Table 4: slopes

$\gamma_{in} = 2(\beta - 3/2)/(\beta - 1/2)$ and $\gamma_{out} = \alpha + 3/2$ are monotone in one parameter,
so their bootstrap percentiles are the parameter's; the error is the larger half-interval, as in Table 3.
$\gamma(R_{vir})$ is the model's log slope at $r = c\,r_s$. $\gamma_{out}$ is formal:
no form reaches it (check 3). It is the slope of a tracer in a Keplerian potential; a self-consistent
power law would instead need $2n/(n-1)$ with $n = \alpha + 3/2 > 5$, and here $n < 5$, so the models truncate.

In [ ]:
def pm(c, p16, p84, f=lambda v: v):
    return max(abs(f(c) - f(p16)), abs(f(p84) - f(c)))


g_inner = lambda b: 2 * (b - 1.5) / (b - 0.5)
rows = [dict(form="NFW", gamma_in="1", gamma_Rvir=f"{(1 + 3 * C_VIR) / (1 + C_VIR):.2f}", gamma_out="3")]
for k, (t, p) in enumerate(FORMS.items(), 1):
    r = P.loc[k]
    bc = "beta" if k == 2 else "alpha"
    rows.append(dict(
        form=str(k),
        gamma_in=f"{g_inner(r[bc]):.3f} ± {pm(r[bc], r[bc + '_p16'], r[bc + '_p84'], g_inner):.3f}",
        gamma_Rvir=f"{np.interp(np.log(C_VIR), MOD[t]['s'], MOD[t]['slope']):.2f}",
        gamma_out="–" if p["e0"] else f"{r['alpha'] + 1.5:.2f} ± {pm(r['alpha'], r['alpha_p16'], r['alpha_p84']):.2f}"))
TABLE = pd.DataFrame(rows)
TABLE.to_csv(OUT / "slopes_table.csv", index=False)
tex = lambda v: "--" if v == "–" else "$" + v.replace("±", r"\pm") + "$"
(OUT / "slopes_table.tex").write_text("".join(
    f"{x.form} & {tex(x.gamma_in)} & {tex(x.gamma_Rvir)} & {tex(x.gamma_out)} \\\\\n" for x in TABLE.itertuples()))
print(f"c = R_200m / r_s = {C_VIR:.2f}\n")
print(TABLE.to_string(index=False))

## Figure

Dotted vertical line at $r_s$, dashed at Halo 2's $R_{vir}$. In (c), dashed horizontal lines are each form's $\gamma_{in}$.

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 10), gridspec_kw=dict(hspace=0.28, wspace=0.22))
ax = ax.ravel()
panels = [("rho", r"$\rho/\rho_s$", "(a) Density"), ("sigma", r"$\hat\sigma_r$", r"(b) $\hat\sigma_r$"),
          ("slope", r"$-d\ln\rho/d\ln r$", "(c) Log slope"), ("beta", r"$\beta(r)$", r"(d) Anisotropy $\beta(r)$")]
for a, (key, ylab, title) in zip(ax, panels):
    for h in HALOS:
        a.plot(DATA[h]["x"], DATA[h][key], color=HCOL[h], lw=2.4 if h == REF else 1.3, label=f"Halo {h:02d}")
    if key == "beta":
        a.axhline(0, color=FCOL["form1"], lw=2.4)   # the models are isotropic
    else:
        for t, m in MOD.items():
            w = inner(m)
            a.plot(m["x"][w], m[key][w], color=FCOL[t], lw=2.4, label=f"Form {t[-1]}")
    a.set(xscale="log", xlim=(lo, X_MAX), xlabel=r"$r/r_s$", ylabel=ylab)
    a.set_title(title, loc="left", fontsize=11)
    a.axvline(1, color="k", lw=0.8, ls=":")
    a.axvline(C_VIR, color="k", lw=0.8, ls="--")

ax[0].plot(xg, nfw(xg), "k--", lw=1.5, label="NFW")
ax[0].plot(xg, einasto(xg), "k:", lw=1.5, label="Einasto")
ax[0].set_yscale("log")
ax[0].legend(fontsize=9)
ax[2].plot(xg, logslope(nfw, xg), "k--", lw=1.5)
ax[2].plot(xg, logslope(einasto, xg), "k:", lw=1.5)
for t, p in FORMS.items():
    ax[2].axhline(g_inner(p["b"]), color=FCOL[t], lw=1.4, ls="--")
for t, v in KEP.items():
    ax[2].axhline(v, color=FCOL[t], lw=1.4, ls=":")
for k in (1, 2, 3):
    ax[2].axhline(k, color="0.6", lw=0.8, ls=":")
ax[2].set_ylim(0, 4.2)
ax[0].text(C_VIR / 1.06, 0.27, r"$R_{\rm vir}$ (Halo 2)", rotation=90, ha="right", va="bottom", fontsize=9,
           transform=ax[0].get_xaxis_transform())
ax[3].set_ylim(-0.5, 0.8)

fig.savefig(OUT / "fig7_preview.png", dpi=150, bbox_inches="tight")
plt.show()

## Export: .dat files and the gnuplot script

Paper style (cairolatex). Same content as the matplotlib figure.

In [ ]:
for t, m in MOD.items():
    w = inner(m)
    np.savetxt(OUT / f"{t}.dat", np.column_stack([m["x"][w], m["rho"][w], m["sigma"][w], m["slope"][w]]),
               header="x rho_hat sigma_r slope")
for h, d in DATA.items():
    np.savetxt(OUT / f"halo{h:02d}.dat", np.column_stack([d["x"], d["rho"], d["sigma"], d["slope"], d["beta"]]),
               header="x rho_hat sigma_r slope beta")
np.savetxt(OUT / "profiles.dat", np.column_stack([xg, nfw(xg), einasto(xg), logslope(nfw, xg), logslope(einasto, xg)]),
           header="x nfw einasto nfw_slope einasto_slope")

hi = X_MAX
g_in = {t: g_inner(p["b"]) for t, p in FORMS.items()}

B = "\\\\large\\\\bfseries "


def items(col, mcol, keys):
    out = [f'"halo{h:02d}.dat" using 1:{col} with lines lc rgb "{HCOL[h]}" lw {4.0 if h == REF else 2.2} '
           + (f'title "{{{B}Halo {h:02d}}}"' if keys else "notitle") for h in HALOS]
    if mcol:
        out += [f'"{t}.dat" using 1:{mcol} with lines lc rgb "{FCOL[t]}" lw 4.0 '
                + (f'title "{{{B}Form {t[-1]}}}"' if keys else "notitle") for t in FORMS]
    return out


def plot(lines):
    return "plot \\\n  " + ", \\\n  ".join(lines)


panel = lambda l, r, b, t: (f"set lmargin at screen {l}\nset rmargin at screen {r}\n"
                            f"set bmargin at screen {b}\nset tmargin at screen {t}\n")
vlines = ('set arrow 1 from 1, graph 0 to 1, graph 1 nohead front lc rgb "#000000" lw 1.2 dt 3\n'
          f'set arrow 2 from {C_VIR:.5g}, graph 0 to {C_VIR:.5g}, graph 1 nohead front lc rgb "#000000" lw 1.2 dt 2\n')
rvir_label = lambda y: (rf'set label 9 "{{\\normalsize $R_{{\\rm vir}}$ (Halo 2)}}" at {C_VIR * 1.11:.5g}, graph {y} '
                         'left rotate by 90 front\n')

PLOT_A = plot(items(2, 2, True) + ['"profiles.dat" using 1:2 with lines ls 17 title "{' + B + 'NFW}"',
                           '"profiles.dat" using 1:3 with lines ls 18 title "{' + B + 'Einasto}"'])
PLOT_B = plot(items(3, 3, False))
PLOT_C = plot(items(4, 4, False) + ['"profiles.dat" using 1:4 with lines ls 17 notitle',
                            '"profiles.dat" using 1:5 with lines ls 18 notitle']
      + [f'{v:.5g} with lines lc rgb "{FCOL[t]}" lw 2.6 dt 2 notitle' for t, v in g_in.items()]
      + [f'{v:.5g} with lines lc rgb "{FCOL[t]}" lw 2.6 dt 3 notitle' for t, v in KEP.items()]
      + [f'{k} with lines lc rgb "#808080" dt 3 lw 1.8 notitle' for k in (1, 2, 3)])
PLOT_D = plot(items(5, 0, False) + ['0 with lines lc rgb "#ff7f0e" lw 4.0 notitle'])

gnu = rf"""# Fig. 7, written by fig7_table4_physical_consequences.ipynb. Render on the Mac:
#   gnuplot fig7.gnu        (gnuplot and pdflatex on PATH)
reset
set encoding utf8
set datafile commentschars "#"
set term cairolatex pdf standalone color size 13in,10in font ",12" \
    header "\\usepackage[margin=0.12in]{{geometry}}\\usepackage{{amssymb}}\\usepackage{{amsmath}}"
set output "fig7.tex"
set multiplot
set border 3 lw 2.5
set tics out nomirror scale 1.2
set ytics offset char -0.8, 0
set style line 17 lc rgb "#000000" lw 3.0 dt 2
set style line 18 lc rgb "#000000" lw 3.0 dt 3
set style line 23 lc rgb "#000000" lw 2.5 dt 3
set logscale x
set xrange [{lo:.5g}:{hi:.5g}]
set format x '{{\large $10^{{%L}}$}}'
set xlabel "{{\\Large \\boldmath$r/r_s$}}"

# (a) density
{panel(0.08, 0.47, 0.53, 0.92)}set logscale y
set grid back lt 0 lc rgb "#d0d0d0"
set format y '{{\large $10^{{%L}}$}}'
set ylabel "{{\\LARGE \\boldmath$\\rho/\\rho_s$}}" offset char 1.5, 0
set key top right spacing 1.35 samplen 2.2
{vlines}{rvir_label(0.37)}set label 1 "{{\\large\\bfseries (a) Density profile}}" at graph 0.02, graph 0.95 left front
{PLOT_A}

# (b) dispersion
{panel(0.53, 0.92, 0.53, 0.92)}unset logscale y
unset grid
unset key
unset label 1
unset label 9
unset arrow
set format y '{{\large %g}}'
set ylabel "{{\\LARGE \\boldmath$\\hat{{\\sigma}}_r$}}" offset char 1.5, 0
set label 2 "{{\\large\\bfseries (b) \\boldmath$\\hat{{\\sigma}}_r$}}" at graph 0.02, graph 0.95 left front
{PLOT_B}

# (c) log slope
{panel(0.08, 0.47, 0.08, 0.47)}set grid back lt 0 lc rgb "#d0d0d0"
unset label 2
unset arrow
set yrange [0:4.2]
set ylabel "{{\\LARGE \\boldmath$-d\\ln\\rho/d\\ln r$}}" offset char 1.5, 0
{vlines}set label 3 "{{\\large\\bfseries (c) Log slope}}" at graph 0.02, graph 0.95 left front
{PLOT_C}

# (d) anisotropy
{panel(0.53, 0.92, 0.08, 0.47)}unset grid
unset label 3
unset arrow
set yrange [-0.5:0.8]
set ylabel "{{\\LARGE \\boldmath$\\beta(r)$}}" offset char 1.5, 0
set label 4 "{{\\large\\bfseries (d) Anisotropy \\boldmath$\\beta(r)$}}" at graph 0.02, graph 0.95 left front
{PLOT_D}

unset multiplot
unset output
!sed -i '' 's/\\makebox(0,0){{\\large\\bfseries (/\\makebox(0,0)[l]{{\\large\\bfseries (/g' fig7.tex
!sed -i '' 's/\\makebox(0,0){{\\large\\bfseries/\\makebox(0,0)[r]{{\\large\\bfseries/g' fig7.tex
!pdflatex -interaction=nonstopmode fig7.tex > /dev/null
!rm -f fig7.aux fig7.log fig7-inc.pdf
!gs -dSAFER -dBATCH -dNOPAUSE -dTextAlphaBits=4 -dGraphicsAlphaBits=4 -r300 -sDEVICE=png16m -sOutputFile=fig7.png fig7.pdf > /dev/null
print "Wrote fig7.pdf and fig7.png"
"""
(OUT / "fig7.gnu").write_text(gnu)
print(f"wrote {OUT}/")